<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
!pip install -q tensorflow-recommenders tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.6 MB/s eta 0:00:00


In [3]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [6]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"
FEATURE_PATH = "/content/drive/MyDrive/Recommendation_Engine/features"
VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

In [7]:
customers_df = spark.read.parquet(f"{PROCESSED_PATH}/customers_clean.parquet")

articles_df = spark.read.parquet(f"{PROCESSED_PATH}/articles_clean.parquet")

transactions_df = spark.read.parquet(f"{PROCESSED_PATH}/transactions_clean.parquet")

In [8]:
recency_df = spark.read.parquet(f"{FEATURE_PATH}/recency.parquet")

product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/product_popularity.parquet"
)

monthly_product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/monthly_product_popularity.parquet"
)

In [9]:
customer_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

product_type_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/product_type_vocab.parquet"
)

department_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/department_vocab.parquet"
)

color_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/color_vocab.parquet"
)

In [10]:
print("Processed Datasets")
print("------------------")
print("Customers      :", customers_df.count())
print("Articles       :", articles_df.count())
print("Transactions   :", transactions_df.count())

print("\nFeature Tables")
print("------------------")
print("Recency                :", recency_df.count())
print("Product Popularity     :", product_popularity_df.count())
print("Monthly Popularity     :", monthly_product_popularity_df.count())

print("\nVocabularies")
print("------------------")
print("Customer Vocabulary    :", customer_vocab.count())
print("Article Vocabulary     :", article_vocab.count())
print("Product Type Vocabulary:", product_type_vocab.count())
print("Department Vocabulary  :", department_vocab.count())
print("Color Vocabulary       :", color_vocab.count())

Processed Datasets
------------------
Customers      : 1371980
Articles       : 105542
Transactions   : 31788324

Feature Tables
------------------
Recency                : 1362281
Product Popularity     : 104547
Monthly Popularity     : 768883

Vocabularies
------------------
Customer Vocabulary    : 1371980
Article Vocabulary     : 105542
Product Type Vocabulary: 131
Department Vocabulary  : 250
Color Vocabulary       : 50


#Create the Interaction Dataset

In [11]:
interactions_df = transactions_df.select(
    "customer_id",
    "article_id"
)

print("Total Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Total Interactions: 31788324
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016003 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016001 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|682236013 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016016 |
|aaa7a0483dd5b9e395d95324dcbfeb617af9800f39487d4b6aaee662bcd384c7|783335003 |
|aaa7b371465a823fec4312ef0f2807f924d54e5d41afb686223b76266bd9c599|563519008 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|783056001 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|695325016 |
|aaa8f491632b9022bf20aa444c793bdf23621bd9463c050e86b73ada4cb059b6|757333001 |
|aaa8f491632b9022bf20aa444c793bdf23

#Remove Duplicate User–Item Pairs

In [12]:
interactions_df = interactions_df.dropDuplicates()

print("Unique User-Item Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Unique User-Item Interactions: 27306439
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaac535f79b71437632d6001ebd960766da3000d7c455cd7a0500dab24cfd50e|799507001 |
|abd2537848661862039af47c8bcee2f9620c5fdc3b13c1af1f66f1f485c51de7|399087021 |
|ac2a5d7aa83653f77dbb343a90ebb705fb3c0a1e2683dbf74b1e08dab042c9bd|821152001 |
|ac43e628b476ec53ee48233d5ff7ad26d91b114095b51c8bdf8f5b560d101218|835730001 |
|acd03ec982613dcc026b69a4323f1db291cfbcd6a20173fab14c1063b7c014f2|708473003 |
|acfcd9df9a2a130cc54f547ea5f5829c5ff1913c9fda8e5b29a7badcbf544e26|737222004 |
|ad410adca76cb24d968194c0c2cf02d4714c2b8a639c9377c61020dc1972e8ef|734623002 |
|adb4d1ca1ae86f0a4592ba7ee5d586662945a45bb8d5a76761d971538f2c7980|728703008 |
|adceb8b35d5250062e3bd8b2a5025ee782874c798227c143ef6691488c75fb4d|399223001 |
|adf5b91a4a8092d8f2ce64e

#Create a Training Sample

In [13]:
from pyspark.sql.functions import rand

training_sample = (
    interactions_df
    .orderBy(rand())
    .sample(withReplacement=False, fraction=0.05, seed=42)
)

print("Training Sample Size:", training_sample.count())

training_sample.show(10, truncate=False)

Training Sample Size: 1366014
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|8966dd82c73d09949858f08432b755fb36e17c7e7f381cd64b1fbf7279eebc64|856310003 |
|797c3b797e80d32ad49e95229885f564b357be7ae239593d32b21e3a3206b869|854663005 |
|68f0af25635ec349d5a57a98d2768622b285567c1fc39e399f034fa08e27b7c5|573937001 |
|22f8248b4d9f397a9c12211ecd88c933774511514a4cc4a5f3c6ef5e058f61b2|685814022 |
|ce2908b10c452e161b0d2c600a6246b96b39686380a4bde31cae4395632fb7c0|706016003 |
|e7a93b6b8cc4a66f1629acb1f566429fa1ca7a9d42cf91c3bba76b3078c2112a|723347001 |
|ee8f7bb3db4effb7b17756f9f1a65ed9ea941484ea6a150e618ab025ef39ee6d|301656013 |
|f3c5937a4eb7b151d189d553efb70194cac1957dd88ab7826df603c3b5b79628|745977001 |
|ec04acf276d2c2aa216a5ccc55e5e270a11ceffed39a8618690c0502eb269908|685448001 |
|83e482f3b8c3a9e4e354cc453f96cc9af

#Convert to NumPy

In [14]:
training_pd = training_sample.toPandas()

customer_ids = training_pd["customer_id"].astype(str).values
article_ids = training_pd["article_id"].astype(str).values

print(customer_ids[:5])
print(article_ids[:5])

['8966dd82c73d09949858f08432b755fb36e17c7e7f381cd64b1fbf7279eebc64'
 '797c3b797e80d32ad49e95229885f564b357be7ae239593d32b21e3a3206b869'
 '68f0af25635ec349d5a57a98d2768622b285567c1fc39e399f034fa08e27b7c5'
 '22f8248b4d9f397a9c12211ecd88c933774511514a4cc4a5f3c6ef5e058f61b2'
 'ce2908b10c452e161b0d2c600a6246b96b39686380a4bde31cae4395632fb7c0']
['856310003' '854663005' '573937001' '685814022' '706016003']


#Build the TensorFlow Dataset

In [15]:
import tensorflow as tf

interactions_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": customer_ids,
    "article_id": article_ids
})

In [16]:
for sample in interactions_ds.take(5):
    print(sample)

{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'8966dd82c73d09949858f08432b755fb36e17c7e7f381cd64b1fbf7279eebc64'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'856310003'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'797c3b797e80d32ad49e95229885f564b357be7ae239593d32b21e3a3206b869'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'854663005'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'68f0af25635ec349d5a57a98d2768622b285567c1fc39e399f034fa08e27b7c5'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'573937001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'22f8248b4d9f397a9c12211ecd88c933774511514a4cc4a5f3c6ef5e058f61b2'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'685814022'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'ce2908b10c452e161b0d2c600a6246b96b39686380a4bde31cae4395632fb7c0'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'706016003'>}


#Prepare for Training

In [17]:
BATCH_SIZE = 8192

train_ds = (
    interactions_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Create the Lookup Layers

In [18]:
customer_ids_vocab = (
    customer_vocab
    .select("customer_id")
    .toPandas()["customer_id"]
    .astype(str)
    .tolist()
)

article_ids_vocab = (
    article_vocab
    .select("article_id")
    .toPandas()["article_id"]
    .astype(str)
    .tolist()
)

print("Customers:", len(customer_ids_vocab))
print("Articles :", len(article_ids_vocab))

Customers: 1371980
Articles : 105542


#Build the Lookup Layers

In [19]:
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_ids_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_ids_vocab,
    mask_token=None
)

#Test the Lookup

In [20]:
sample_customer = customer_ids[0]
sample_article = article_ids[0]

print("Customer Index:", customer_lookup(tf.constant(sample_customer)).numpy())
print("Article Index :", article_lookup(tf.constant(sample_article)).numpy())

Customer Index: 52993
Article Index : 93033


#Build the Query Tower

In [21]:
query_tower = tf.keras.Sequential([
    customer_lookup,

    tf.keras.layers.Embedding(
        input_dim=customer_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Query Tower

In [22]:
sample_embedding = query_tower(
    tf.constant([customer_ids[0]])
)

print(sample_embedding.shape)
print(sample_embedding.numpy())

(1, 64)
[[ 0.00833868  0.00786999  0.00898826  0.04217878 -0.00397931 -0.03254192
  -0.02089103 -0.00507454  0.00044513  0.00041815 -0.00141257  0.01136817
   0.02360821  0.01294271  0.02057552  0.01076348  0.00187264 -0.02603451
  -0.00784129  0.00368006 -0.00885977  0.00111126 -0.00626916 -0.00706251
  -0.06000421 -0.019649    0.00629772 -0.00095282  0.0008876  -0.01400276
   0.02315518 -0.01541404 -0.00972253  0.00664588 -0.02820265  0.01333619
   0.01482129 -0.01309861  0.00096966  0.03802887 -0.02442697  0.00014253
  -0.01842638 -0.02975116 -0.00475132 -0.00461418  0.0214374  -0.02012909
   0.0140556  -0.04407423 -0.01728641  0.01215209 -0.00776574  0.00461252
   0.01213924 -0.03855959 -0.02675951 -0.00485435  0.010938    0.01573326
   0.01028757  0.00073508  0.02799111  0.00305213]]


#Build the Candidate Tower

In [23]:
candidate_tower = tf.keras.Sequential([
    article_lookup,

    tf.keras.layers.Embedding(
        input_dim=article_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Candidate Tower

In [24]:
sample_item_embedding = candidate_tower(
    tf.constant([article_ids[0]])
)

print(sample_item_embedding.shape)
print(sample_item_embedding.numpy())

(1, 64)
[[ 0.0073253   0.03272892 -0.03203727  0.02082973 -0.03523952 -0.03082554
   0.00251026  0.00017497 -0.0069518   0.02885602 -0.00328968 -0.02410926
  -0.0007774   0.01185731 -0.01086978  0.05361247 -0.00862525  0.00496943
  -0.02755268  0.01928602  0.01098907  0.02103271  0.00968553 -0.01072738
   0.01616484 -0.01660766 -0.01897039  0.03706327 -0.01603188 -0.03178605
  -0.01593032 -0.03755497  0.00719918 -0.03779716  0.03921026  0.0099795
  -0.0157427   0.01679536 -0.00766104  0.02558021 -0.02628678  0.02148171
  -0.0137679   0.00192913  0.02542537  0.01760782  0.02853802  0.01455106
  -0.00031045 -0.01737934 -0.02430839 -0.0030574  -0.01930704 -0.01723484
  -0.01327354 -0.00462503 -0.01041289 -0.02633626  0.01441796  0.02254124
   0.01729221 -0.05611916  0.01754285  0.01564815]]


#Verify Both Towers

In [25]:
print("User Embedding Shape :", sample_embedding.shape)
print("Item Embedding Shape :", sample_item_embedding.shape)

User Embedding Shape : (1, 64)
Item Embedding Shape : (1, 64)


#Create the Candidate Dataset

In [26]:
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(article_ids_vocab)
    .batch(1024)
)

In [27]:
for batch in candidate_dataset.take(1):
    print(batch[:5])

tf.Tensor([b'108775015' b'108775044' b'108775051' b'110065001' b'110065002'], shape=(5,), dtype=string)


#Build the Retrieval Task

In [28]:
import tensorflow_recommenders as tfrs

retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=candidate_dataset.map(candidate_tower)
    )
)

#Build the Complete Recommendation Model

In [29]:
class HMRecommendationModel(tfrs.models.Model):

    def __init__(self, query_model, candidate_model):
        super().__init__()

        self.query_model = query_model
        self.candidate_model = candidate_model

        self.task = retrieval_task

    def compute_loss(self, features, training=False):

        user_embeddings = self.query_model(features["customer_id"])

        item_embeddings = self.candidate_model(features["article_id"])

        return self.task(
            user_embeddings,
            item_embeddings
        )

In [30]:
import os

MODEL_PATH = "/content/drive/MyDrive/Recommendation_Engine/models"

os.makedirs(MODEL_PATH, exist_ok=True)

print("Model folder created.")

Model folder created.


In [31]:
query_tower.save(f"{MODEL_PATH}/query_tower.keras")
candidate_tower.save(f"{MODEL_PATH}/candidate_tower.keras")

print("Model architectures saved.")

Model architectures saved.


In [32]:
import json

config = {
    "embedding_dimension": 64,
    "hidden_layer": 128,
    "batch_size": 8192,
    "sample_fraction": 0.05,
    "tensorflow_version": "2.20.0",
    "tfrs_version": "0.7.7"
}

with open(f"{MODEL_PATH}/model_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved.")

Configuration saved.


Instantiate the Model

In [33]:
model = HMRecommendationModel(
    query_model=query_tower,
    candidate_model=candidate_tower
)

print(model)

#Prepare the Dataset

In [34]:
dataset_size = len(customer_ids)

train_size = int(0.8 * dataset_size)

train_ds = interactions_ds.take(train_size)
test_ds = interactions_ds.skip(train_size)

#Batch the Data

In [35]:
BATCH_SIZE = 8192

train_ds = (
    train_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Verify the Split

In [36]:
print("Dataset Size :", dataset_size)
print("Training Size:", train_size)
print("Testing Size :", dataset_size - train_size)

Dataset Size : 1363824
Training Size: 1091059
Testing Size : 272765


#Compile the Model

In [37]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

#Train the Model

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5,
    verbose=1
)

Epoch 1/5
 29/134 [=====>........................] - ETA: 1:03:07 - factorized_top_k/top_1_categorical_accuracy: 0.0046 - factorized_top_k/top_5_categorical_accuracy: 0.0071 - factorized_top_k/top_10_categorical_accuracy: 0.0083 - factorized_top_k/top_50_categorical_accuracy: 0.0143 - factorized_top_k/top_100_categorical_accuracy: 0.0188 - loss: 73823.8960 - regularization_loss: 0.0000e+00 - total_loss: 73823.8960